In [1]:
# !python -m pip install pandas

In [ ]:
import pandas as pd
import numpy as np

# Cargamos el archivo CSV que está en la misma carpeta
df_weight = pd.read_csv('05 weightLogInfo_merged.csv')

# Le pedimos a Python que nos muestre las primeras 5 filas para comprobar que todo está bien
df_weight.head()

print(df_weight['Id'].nunique())
print()
print(df_weight['Id'].unique())


8

[1503960366 1927972279 2873212765 4319703577 4558609924 5577150313
 6962181067 8877689391]


In [3]:
df_weight['Id'].value_counts()

Id
6962181067    30
8877689391    24
4558609924     5
1503960366     2
2873212765     2
4319703577     2
1927972279     1
5577150313     1
Name: count, dtype: int64

In [4]:
# Muestra los distintos pesos que registró cada ID único
df_weight.groupby('Id')['WeightKg'].unique()

Id
1503960366                                   [52.5999984741211]
1927972279                                              [133.5]
2873212765                 [56.7000007629395, 57.2999992370605]
4319703577                 [72.4000015258789, 72.3000030517578]
4558609924    [69.6999969482422, 70.3000030517578, 69.900001...
5577150313                                   [90.6999969482422]
6962181067    [62.5, 62.0999984741211, 61.7000007629395, 61....
8877689391    [85.8000030517578, 84.9000015258789, 84.5, 85....
Name: WeightKg, dtype: object

In [5]:
# Calcula el BMI stats para cada usuario único
bmi_stats = (
    df_weight.groupby('Id')['BMI']
    .agg(
        mode=lambda x: x.mode().iloc[0] if not x.mode().empty else None,
        std='std'
    )
    .round(2)
    .reset_index()
)
print(bmi_stats)

           Id   mode   std
0  1503960366  22.65  0.00
1  1927972279  47.54   NaN
2  2873212765  21.45  0.17
3  4319703577  27.38  0.05
4  4558609924  27.00  0.19
5  5577150313  28.00   NaN
6  6962181067  23.89  0.15
7  8877689391  25.41  0.14


In [ ]:


# 1. Definimos los límites (bins) de los rangos de BMI
# Usamos np.inf para indicar que el último rango va desde 40 hasta el infinito
limites = [0, 18.5, 25.0, 30.0, 35.0, 40.0, np.inf]

# 2. Definimos las etiquetas exactas que quieres para cada rango
etiquetas = [
    "Bajo peso",
    "Peso normal (saludable)",
    "Sobrepeso",
    "Obesidad-Clase I (Moderada)",
    "Obesidad-Clase II (Severa)",
    "Obesidad-Clase III (Mórbida/Grave)"
]

# 3. Creamos la nueva columna aplicando pd.cut sobre la columna 'mode' de tu bmi_stats
# right=False asegura que si un valor es exactamente 25.0, entre en "Sobrepeso" y no en "Peso normal"

bmi_stats_categories = bmi_stats.copy()

bmi_stats_categories['BMI Categories'] = pd.cut(bmi_stats['mode'], bins=limites, labels=etiquetas, right=False)

# Sort by 'mode' ascending and reset the index tracking
bmi_stats_categories = bmi_stats_categories.sort_values(by='mode', ascending=True).reset_index(drop=True)

# Vemos el resultado final
bmi_stats_categories.head(10)

,Id,mode,std,BMI Categories
0,2873212765,21.45,0.17,Peso normal (saludable)
1,1503960366,22.65,0.00,Peso normal (saludable)
2,6962181067,23.89,0.15,Peso normal (saludable)
3,8877689391,25.41,0.14,Sobrepeso
4,4558609924,27.00,0.19,Sobrepeso
5,4319703577,27.38,0.05,Sobrepeso
6,5577150313,28.00,NaN,Sobrepeso
7,1927972279,47.54,NaN,Obesidad-Clase III (Mórbida/Grave)


In [7]:
# Guardar el DataFrame en un archivo CSV en tu computadora
bmi_stats_categories.to_csv('05_bmi_stats_categories.csv', index=False)